# 🧠 Alzheimer's Disease MRI Classification (SafeResNet-18 + SHAP)
### End-to-End Training & Explainability Pipeline on Google Colab (NVIDIA T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/1PratikSingh11/Alzhiemer-Detection-Model/blob/main/Alzheimer_ResNet18_Training_Colab.ipynb)

This notebook trains the `SafeResNet-18` architecture on the OASIS MRI dataset using **Weighted Cross-Entropy Loss** and **StepLR scheduling**, and extracts pixel-level **SHAP GradientExplainer** attributions.

## 1. Verify GPU Hardware
Check that Google Colab is connected to a dedicated NVIDIA GPU (Tesla T4 / P100).

In [ ]:
!nvidia-smi
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print(f'Active Training Device: {device} ({gpu_name})')

## 2. Clone Repository & Install Dependencies

In [ ]:
!git clone https://github.com/1PratikSingh11/Alzhiemer-Detection-Model.git
%cd Alzhiemer-Detection-Model
!pip install -r requirements.txt

## 3. Dataset Setup (OASIS MRI Dataset)
The full OASIS dataset contains 86,437 scans across 4 severity stages.
You can either download the full dataset from Kaggle or generate synthetic samples for quick verification.

In [ ]:
# Option A: For full training from Kaggle API:
# !pip install -q kaggle
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d ninadaithal/imagesoasis -p data/ --unzip

# Option B: Generate simulated sample cohort:
from data.generate_oasis_mock import generate_dataset
generate_dataset(output_root='data', samples_per_class_train=40, samples_per_class_val=10)

## 4. Train SafeResNet-18 (10 Epochs with StepLR & Class Weights)
- **Architecture**: `SafeResNet18` (in-place ReLUs disabled)
- **Loss**: Weighted Cross-Entropy ($w_c = N / (C \cdot N_c)$)
- **Optimizer**: Adam (`lr = 1e-4`, `weight_decay = 1e-4`)
- **Scheduler**: StepLR (`step_size = 5`, `gamma = 0.1`)

In [ ]:
!python train.py --epochs 10 --batch_size 32 --lr 1e-4 --step_size 5 --gamma 0.1 --device cuda

## 5. Visualize Training Curves & Confusion Matrix

In [ ]:
from IPython.display import Image, display
display(Image('outputs_tcc_resnet18/training_validation_curves.png'))
display(Image('outputs_tcc_resnet18/confusion_matrix_normalized.png'))
display(Image('outputs_tcc_resnet18/multiclass_roc_curves.png'))

## 6. Generate SHAP & Grad-CAM Explainability Maps

In [ ]:
!python shap_explain.py --model_path models/best_model.pth --output_dir shap_analysis